# Rolling Robust Scaling Pipeline

Numba-accelerated rolling robust scaler and walk-forward backtest infrastructure
for online volatility forecasting.

This notebook walks through:
1. Load data
2. Apply robust transforms (diurnal + semantic + winsorization)
3. Generate HAR lag features (imported from `src.transforms`)
4. Apply rolling robust scaling to the feature matrix
5. Ring buffer for walk-forward training data
6. Generic walk-forward backtest loop

The final cell writes the complete `scaling.py` module to `../src/scaling.py`.

In [ ]:
# ── Setup: clone repo, install deps ──
import os
import sys

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q numpy pandas pyarrow numba scikit-learn tqdm

sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# ── Load data + robust transform on RV ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data("all30min")
adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"\nadj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# --- Generate HAR features using shared pipeline ---
from src.transforms import resolve_har_lags, generate_har_features, PERIODS_PER_DAY

df, feature_names = generate_har_features(df, target_col="adj_RV")
har_lags = resolve_har_lags()

# Drop burn-in NaN rows
max_lag = har_lags[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

# Extract numpy arrays
X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)

print(f"Feature matrix: {X.shape}")
print(f"Feature names: {feature_names}")
print(f"\nPre-scaling feature stats:")
print(pd.DataFrame(X, columns=feature_names).describe())

In [ ]:
# export
"""Rolling robust scaler and walk-forward backtest infrastructure.

Numba-accelerated sorted-matrix quantile tracking for O(W) median/IQR
scaling, ring buffer for training data, and generic walk-forward loop.
"""

from __future__ import annotations

from collections.abc import Callable
from typing import Any

import numpy as np
from numba import njit
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Numba kernels
# ---------------------------------------------------------------------------

In [ ]:
# Sanity: imports resolved (numba, tqdm). Nothing to assert for the section header.
print("imports ok:", np.__name__, "numba njit decorator:", njit.__module__)

In [ ]:
# export
@njit(cache=True)
def _update_sorted_matrix(sorted_mat: np.ndarray, x_old: np.ndarray, x_new: np.ndarray) -> None:
    """Replace *x_old* with *x_new* in each feature's sorted window."""
    n_features, w = sorted_mat.shape
    for i in range(n_features):
        v_old = x_old[i]
        v_new = x_new[i]
        idx_old = np.searchsorted(sorted_mat[i], v_old)
        idx_new = np.searchsorted(sorted_mat[i], v_new)
        if idx_old < idx_new:
            idx_new -= 1
            for j in range(idx_old, idx_new):
                sorted_mat[i, j] = sorted_mat[i, j + 1]
        elif idx_old > idx_new:
            for j in range(idx_old, idx_new, -1):
                sorted_mat[i, j] = sorted_mat[i, j - 1]
        sorted_mat[i, idx_new] = v_new

In [ ]:
# Test _update_sorted_matrix on a toy 2-feature window of length 4.
toy_sorted = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 20.0, 30.0, 40.0]])
x_old = np.array([2.0, 20.0])
x_new = np.array([5.0, 5.0])
_update_sorted_matrix(toy_sorted, x_old, x_new)
print("after update:", toy_sorted)
assert np.all(np.diff(toy_sorted, axis=1) >= 0), "rows must stay sorted"

In [ ]:
# export
@njit(cache=True)
def _get_robust_stats(
    sorted_mat: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute median and IQR from pre-sorted rolling window."""
    n_features, w = sorted_mat.shape
    median = np.empty(n_features, dtype=np.float64)
    iqr = np.empty(n_features, dtype=np.float64)
    idx_25 = (w - 1) * 0.25
    idx_50 = (w - 1) * 0.50
    idx_75 = (w - 1) * 0.75
    i25_floor, rem_25 = int(idx_25), idx_25 - int(idx_25)
    i50_floor, rem_50 = int(idx_50), idx_50 - int(idx_50)
    i75_floor, rem_75 = int(idx_75), idx_75 - int(idx_75)
    for i in range(n_features):
        q25 = sorted_mat[i, i25_floor] * (1.0 - rem_25) + sorted_mat[i, min(i25_floor + 1, w - 1)] * rem_25
        med = sorted_mat[i, i50_floor] * (1.0 - rem_50) + sorted_mat[i, min(i50_floor + 1, w - 1)] * rem_50
        q75 = sorted_mat[i, i75_floor] * (1.0 - rem_75) + sorted_mat[i, min(i75_floor + 1, w - 1)] * rem_75
        median[i] = med
        iq = q75 - q25
        iqr[i] = iq if iq >= 1e-12 else 1.0
    return median, iqr

In [ ]:
# Test _get_robust_stats against numpy reference on a small sorted window.
rng = np.random.default_rng(0)
raw = rng.normal(size=(3, 21))
sorted_win = np.sort(raw, axis=1)
med, iqr = _get_robust_stats(sorted_win)
ref_med = np.median(raw, axis=1)
ref_iqr = np.percentile(raw, 75, axis=1) - np.percentile(raw, 25, axis=1)
print("median:", med, "ref:", ref_med)
print("iqr:   ", iqr, "ref:", ref_iqr)
assert np.allclose(med, ref_med)
assert np.allclose(iqr, ref_iqr)

In [ ]:
# export
# ---------------------------------------------------------------------------
# Rolling robust scaler
# ---------------------------------------------------------------------------


class RollingRobustScaler:
    """Online robust scaler backed by sorted-matrix quantile tracking.

    Maintains a rolling window of observations and a parallel sorted matrix
    per feature for O(W) median/IQR computation.  Supports two usage styles:

    1. **get_scaler** -- return (median, iqr) for manual scaling (ridge-style).
    2. **transform_single / transform_buffer** -- scale directly (PCR-style).

    Parameters
    ----------
    window_size : int
        Rolling window length.
    n_features : int, optional
        Number of features.  If given, buffers are pre-allocated;
        otherwise allocation is deferred to :meth:`initialize`.
    """

    buffer: np.ndarray | None
    sorted_mat: np.ndarray | None

    def __init__(self, window_size: int, n_features: int | None = None) -> None:
        self.window_size = window_size
        if n_features is not None:
            self.buffer = np.zeros((window_size, n_features), dtype=np.float64)
            self.sorted_mat = np.zeros((n_features, window_size), dtype=np.float64)
        else:
            self.buffer = None
            self.sorted_mat = None
        self.pos: int = 0

    def initialize(self, data_block: np.ndarray) -> None:
        """Fill buffers from *data_block* ``(window_size, n_features)``."""
        w = self.window_size
        n_features = data_block.shape[1]
        if self.buffer is None:
            self.buffer = np.empty((w, n_features), dtype=np.float64)
            self.sorted_mat = np.empty((n_features, w), dtype=np.float64)
        self.buffer[:] = data_block[:w]
        for i in range(n_features):
            self.sorted_mat[i] = np.sort(data_block[:w, i])  # type: ignore[index]
        self.pos = 0

    def update(self, x_new: np.ndarray) -> None:
        """Slide window: replace oldest row with *x_new*."""
        assert self.buffer is not None and self.sorted_mat is not None
        x_old = self.buffer[self.pos].copy()
        self.buffer[self.pos] = x_new
        _update_sorted_matrix(self.sorted_mat, x_old, x_new)
        self.pos = (self.pos + 1) % self.window_size

    def get_scaler(self) -> tuple[np.ndarray, np.ndarray]:
        """Return ``(median, iqr)`` arrays from the current sorted buffer."""
        assert self.sorted_mat is not None
        return _get_robust_stats(self.sorted_mat)

    def transform_single(self, x: np.ndarray) -> np.ndarray:
        """Scale a single observation using current median/IQR."""
        median, iqr = self.get_scaler()
        return (x - median) / iqr

    def transform_buffer(self) -> np.ndarray:
        """Scale the entire current buffer using current median/IQR."""
        assert self.buffer is not None
        median, iqr = self.get_scaler()
        return (self.buffer - median) / iqr

In [ ]:
# Test RollingRobustScaler on the HAR feature matrix X.
scaler = RollingRobustScaler(window_size=500, n_features=X.shape[1])
scaler.initialize(X[:500])
med0, iqr0 = scaler.get_scaler()
print("initial median:", np.round(med0, 4))
print("initial iqr:   ", np.round(iqr0, 4))

# slide one step and check stats still finite
scaler.update(X[500])
med1, iqr1 = scaler.get_scaler()
assert np.all(np.isfinite(med1)) and np.all(iqr1 > 0)

# transform_single on the next row
x_scaled = scaler.transform_single(X[501])
print("scaled row 501:", np.round(x_scaled, 3))

In [ ]:
# export
# ---------------------------------------------------------------------------
# Rolling buffer
# ---------------------------------------------------------------------------


class RollingBuffer:
    """Ring buffer for (X, y) pairs used in walk-forward backtests."""

    def __init__(self, window_size: int, n_features: int, n_targets: int = 1) -> None:
        self.window_size = window_size
        self.X = np.zeros((window_size, n_features), dtype=np.float64)
        self.y = np.zeros((window_size, n_targets), dtype=np.float64)
        self.pos = 0
        self.count = 0

    def add(self, x_new: np.ndarray, y_new: np.ndarray) -> None:
        self.X[self.pos] = x_new
        self.y[self.pos] = y_new
        self.pos = (self.pos + 1) % self.window_size
        self.count = min(self.count + 1, self.window_size)

    def get_view(self) -> tuple[np.ndarray, np.ndarray]:
        if self.count < self.window_size:
            return self.X[: self.count], self.y[: self.count]
        idx = np.roll(np.arange(self.window_size), -self.pos)
        return self.X[idx], self.y[idx]

In [ ]:
# Test RollingBuffer: before-full and after-full views.
buf = RollingBuffer(window_size=5, n_features=X.shape[1], n_targets=1)
for i in range(3):
    buf.add(X[i], y[i : i + 1])
Xv, yv = buf.get_view()
print("partial fill:", Xv.shape, yv.shape)
assert Xv.shape == (3, X.shape[1])

for i in range(3, 8):
    buf.add(X[i], y[i : i + 1])
Xv, yv = buf.get_view()
print("after wrap:  ", Xv.shape, yv.shape)
assert Xv.shape == (5, X.shape[1])
# oldest row should be the one added at i=3 (chronological order)
assert np.allclose(Xv[0], X[3])

In [ ]:
# export
# ---------------------------------------------------------------------------
# Walk-forward backtest
# ---------------------------------------------------------------------------


def run_backtest(
    model_fn: Callable[[], Any],
    X: np.ndarray,
    y: np.ndarray,
    train_win: int,
    refit_frequency: int = 1,
    use_scaling: bool = True,
) -> np.ndarray:
    """Walk-forward backtest with periodic refit.

    Parameters
    ----------
    model_fn : Callable[[], Any]
        Zero-argument factory returning a fresh model exposing ``.fit(X, y)``
        and ``.predict(X)``. Called once for the initial fit and again at
        every refit step.
    X : np.ndarray
        Feature matrix of shape ``(n_samples, n_features)``.
    y : np.ndarray
        Target vector of shape ``(n_samples,)``.
    train_win : int
        Rolling training window size, in samples.
    refit_frequency : int, default=1
        Cadence (in steps) at which the model is refit on the latest window.
        ``1`` refits every step; larger values amortize fit cost.
    use_scaling : bool, default=True
        If True, apply ``RollingRobustScaler`` (median / IQR) to features
        using statistics computed only over the trailing ``train_win`` window.
        When the feature matrix has been pre-scaled whole-series by
        :func:`rolling_robust_scale` (the chunking-safe path), pass False.

    Returns
    -------
    np.ndarray
        Predictions of shape ``(n_samples - train_win,)``. Entry ``k``
        corresponds to the prediction for sample ``t = train_win + k``.

    Notes
    -----
    **Refit cadence invariant.** A model is refit when
    ``(t - train_win + 1) % refit_frequency == 0``, where ``t`` is the
    current step index in ``[train_win, n_samples)``. This is an amortized
    schedule: cheap models like Ridge typically use ``refit_frequency=1``
    (refit every step), while heavier tree models (XGBoost, LightGBM) use
    larger values to amortize fitting cost across multiple predictions.

    **Strict causality.** At step ``t`` the model is trained on the closed-
    open window ``[t - train_win : t]`` and predicts ``y[t]``. The feature
    row ``X[t]`` is scaled using statistics from ``[t - train_win : t]``
    (the scaler is updated *after* the prediction at step ``t`` is made),
    and ``y[t]`` is appended to the training buffer *after* prediction.
    No look-ahead is possible: target and same-step feature information
    never enter the model used to produce the prediction at ``t``.

    Scaling, when enabled, uses ``RollingRobustScaler`` -- a rolling median
    and inter-quartile range computed over the trailing ``train_win``
    window via numba-accelerated sorted-matrix tracking.

    **Chunking note.** With ``use_scaling=True`` the training buffer stores
    *already-scaled* rows, and the scaler itself looks back ``train_win``
    rows -- so the buffer's effective look-back is ``2 * train_win`` and a
    chunked walk-forward would need a ``2 * train_win`` halo. Pre-scaling
    the whole series with :func:`rolling_robust_scale` and passing
    ``use_scaling=False`` collapses that back to a ``train_win`` halo.
    """
    n_samples, n_features = X.shape
    predictions = np.full(n_samples - train_win, np.nan)

    # Initialize scaler + buffer
    scaler_obj = RollingRobustScaler(train_win, n_features) if use_scaling else None
    buf = RollingBuffer(train_win, n_features, n_targets=1)

    X_init = X[:train_win].copy()
    y_init = y[:train_win].copy()

    if use_scaling:
        assert scaler_obj is not None  # mypy: scaler_obj is non-None when use_scaling=True
        scaler_obj.initialize(X_init)
        med, iqr = scaler_obj.get_scaler()
        X_scaled_init = (X_init - med) / iqr
    else:
        X_scaled_init = X_init

    for i in range(train_win):
        buf.add(X_scaled_init[i], y_init[i : i + 1])

    # Initial fit
    X_buf, y_buf = buf.get_view()
    model = model_fn()
    model.fit(X_buf, y_buf.ravel())

    # Walk forward
    for t in tqdm(range(train_win, n_samples), desc="backtest"):
        x_t_raw = X[t]

        # Scale
        if use_scaling:
            assert scaler_obj is not None  # mypy: scaler_obj is non-None when use_scaling=True
            med, iqr = scaler_obj.get_scaler()
            x_t_scaled = (x_t_raw - med) / iqr
        else:
            x_t_scaled = x_t_raw

        # Predict
        predictions[t - train_win] = model.predict(x_t_scaled.reshape(1, -1))[0]

        # Update scaler
        if use_scaling:
            assert scaler_obj is not None  # mypy: scaler_obj is non-None when use_scaling=True
            scaler_obj.update(x_t_raw)

        # Add to buffer
        buf.add(x_t_scaled, y[t : t + 1])

        # Refit
        if (t - train_win + 1) % refit_frequency == 0:
            X_buf, y_buf = buf.get_view()
            model = model_fn()
            model.fit(X_buf, y_buf.ravel())

    return predictions


def rolling_robust_scale(X: np.ndarray, train_win: int) -> np.ndarray:
    """Per-row rolling robust scaling, computed whole-series.

    Returns ``X`` rescaled exactly as ``run_backtest(..., use_scaling=True)``
    scales it internally: rows ``[0, train_win)`` by the initial window's
    median / IQR, and each row ``t >= train_win`` by the trailing
    ``[t - train_win : t)`` window. It simply replays
    :class:`RollingRobustScaler` over the whole series, so

        run_backtest(rolling_robust_scale(X, w), y, w, rf, use_scaling=False)
        == run_backtest(X, y, w, rf, use_scaling=True)

    bit-for-bit.

    The point is *chunking*. With the in-loop scaler the training buffer
    holds already-scaled rows and the scaler looks back ``train_win`` -- so
    the buffer depends on ``2 * train_win`` rows of history. Pre-scaling
    here moves that look-back into a whole-series transform (alongside HAR /
    winsor / diurnal), so a chunked walk-forward over the pre-scaled matrix
    is fungible with just a ``train_win`` halo.
    """
    X = np.asarray(X, dtype=np.float64)
    n_samples = len(X)
    if train_win > n_samples:
        raise ValueError(f"train_win ({train_win}) exceeds series length ({n_samples})")
    scaler = RollingRobustScaler(train_win, X.shape[1])
    scaler.initialize(X[:train_win])
    out = np.empty_like(X)
    med, iqr = scaler.get_scaler()
    out[:train_win] = (X[:train_win] - med) / iqr
    for t in range(train_win, n_samples):
        med, iqr = scaler.get_scaler()
        out[t] = (X[t] - med) / iqr
        scaler.update(X[t])
    return out

In [ ]:
# Smoke-test run_backtest with a trivial mean-predictor on a small slice.
from sklearn.linear_model import Ridge

small_n = 600
train_win = 500
X_s = X[:small_n]
y_s = y[:small_n]

preds = run_backtest(lambda: Ridge(alpha=1.0), X_s, y_s, train_win=train_win, refit_frequency=10)
print("preds shape:", preds.shape)
print("first 5 preds:", preds[:5])
assert preds.shape == (small_n - train_win,)
assert np.all(np.isfinite(preds))

In [ ]:
import sys

sys.path.insert(0, "..")
from _exporter import export_notebook

export_notebook("04_scaling.ipynb", "../../src/scaling.py")